# 🏥 Medicare Inpatient — Decision Support Tool
### DSCI 5260 · Group 6 · Capstone

---

This tool combines all four research questions into one interactive workflow:

| Panel | RQ | What it does |
|---|---|---|
| 1 | RQ2 | Forecasts expected discharge volume for a hospital–DRG pair |
| 2 | RQ3 | Predicts expected Medicare payment per discharge |
| 3 | RQ3 | Flags if actual payment received is an outlier |

**Run all cells top to bottom → then use the interactive tool in the last cell.**

In [1]:
# ============================================================
# CELL 1 — Install / import libraries
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import ipywidgets
except ImportError:
    install('ipywidgets')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print('✅ All libraries loaded successfully')

✅ All libraries loaded successfully


In [2]:
# ============================================================
# CELL 2 — Load models and data
# ============================================================
# ── Paths — adjust if your folder structure is different ──
DATA_DIR = Path('..') / 'Data' / 'Processed_Data'

print('Loading models and data...')

# Models
rq2_model     = joblib.load(DATA_DIR / 'rq2_xgb_model.pkl')
rq3_model     = joblib.load(DATA_DIR / 'rq3_xgb_model.pkl')

# Target encoding lookups
hosp_te_lookup = joblib.load(DATA_DIR / 'hosp_te_lookup.pkl')
drg_te_lookup  = joblib.load(DATA_DIR / 'drg_te_lookup.pkl')

# Clean dataset (for dropdowns + historical context charts)
df = pd.read_parquet(DATA_DIR / 'df_medidata_clean.parquet')

# Ensure derived columns exist
if 'Payment_Gap' not in df.columns:
    df['Payment_Gap'] = df['Avg_Submtd_Cvrd_Chrg'] - df['Avg_Mdcr_Pymt_Amt']
if 'Payment_Ratio' not in df.columns:
    df['Payment_Ratio'] = df['Avg_Mdcr_Pymt_Amt'] / df['Avg_Submtd_Cvrd_Chrg']

# Build dropdown options
hospital_options = sorted(df['Rndrng_Prvdr_CCN'].dropna().unique().astype(int).tolist())
drg_options      = sorted(df['DRG_Cd'].dropna().unique().astype(int).tolist())

# Average fallbacks
avg_hosp_te = float(hosp_te_lookup.mean())
avg_drg_te  = float(drg_te_lookup.mean())
avg_drg_wt  = float(df.groupby('DRG_Cd')['DRG_Weight'].mean().mean())

# Model performance constants
MAE_RQ2  = 12       # discharges
MAE_RQ3  = 2190.0  # dollars
MAPE_RQ2 = 0.313
MAPE_RQ3 = 0.143

print(f'✅ Models loaded')
print(f'   Hospitals available : {len(hospital_options):,}')
print(f'   DRGs available      : {len(drg_options):,}')
print(f'   Records in dataset  : {len(df):,}')

# ── Hospital lookup dictionary (CCN → attributes) ──
hosp_lookup = (
    df.groupby('Rndrng_Prvdr_CCN')
      .agg(
          Ownership_Type = ('Ownership_Type', lambda x: x.mode()[0] if len(x) > 0 else 'Non-Profit'),
          RUCA_Group     = ('RUCA_Group',     lambda x: x.mode()[0] if len(x) > 0 else 'Metropolitan'),
          BED_CNT        = ('BED_CNT',        'median'),
      )
      .to_dict('index')
)

# ── DRG code → description lookup ──
drg_desc_lookup = (
    df.groupby('DRG_Cd')['DRG_Desc']
      .agg(lambda x: x.mode()[0] if len(x) > 0 else 'Unknown')
      .to_dict()
)

# ── DRG dropdown options with description ──
drg_display_options = {
    str(d): f"{d} — {drg_desc_lookup.get(d, 'Unknown')[:60]}"
    for d in drg_options
}

print(f'✅ Hospital lookup built: {len(hosp_lookup):,} hospitals')
print(f'✅ DRG description lookup built: {len(drg_desc_lookup):,} DRGs')
print(f'   Example DRG 470: {drg_desc_lookup.get(470, "not found")}')


Loading models and data...


✅ Models loaded
   Hospitals available : 3,301
   DRGs available      : 633
   Records in dataset  : 1,178,405
✅ Hospital lookup built: 3,301 hospitals
✅ DRG description lookup built: 633 DRGs
   Example DRG 470: MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTACHMENT OF LOWER EXTREMITY WITHOUT MCC


In [3]:
# ============================================================
# CELL 3 — Feature lists (must match training exactly)
# ============================================================
RQ2_FEATURES = [
    'DRG_Weight',
    'BED_CNT',
    'hosp_te',
    'drg_te',
    'own_For-Profit',
    'own_Non-Profit',
    'ruca_Metropolitan',
    'ruca_Micropolitan',
    'ruca_Small Town',
    'Data_Year',
]

RQ3_FEATURES = [
    'DRG_Weight',
    'BED_CNT',
    'Log_Tot_Dschrgs',
    'own_For-Profit',
    'own_Non-Profit',
    'ruca_Metropolitan',
    'ruca_Micropolitan',
    'ruca_Small Town',
    'Data_Year',
    'outlier_payment_flag',
]

print('✅ Feature lists confirmed')
print(f'   RQ2 features : {RQ2_FEATURES}')
print(f'   RQ3 features : {RQ3_FEATURES}')

✅ Feature lists confirmed
   RQ2 features : ['DRG_Weight', 'BED_CNT', 'hosp_te', 'drg_te', 'own_For-Profit', 'own_Non-Profit', 'ruca_Metropolitan', 'ruca_Micropolitan', 'ruca_Small Town', 'Data_Year']
   RQ3 features : ['DRG_Weight', 'BED_CNT', 'Log_Tot_Dschrgs', 'own_For-Profit', 'own_Non-Profit', 'ruca_Metropolitan', 'ruca_Micropolitan', 'ruca_Small Town', 'Data_Year', 'outlier_payment_flag']


In [4]:
# ============================================================
# CELL 4 — Helper functions
# ============================================================

def build_rq2_row(hospital_ccn, drg_code, bed_count, ownership, location, year,
                   hosp_te_val, drg_te_val, drg_wt_val):
    return pd.DataFrame([{
        'DRG_Weight'        : drg_wt_val,
        'BED_CNT'           : bed_count,
        'hosp_te'           : hosp_te_val,
        'drg_te'            : drg_te_val,
        'own_For-Profit'    : 1 if ownership == 'For-Profit'   else 0,
        'own_Non-Profit'    : 1 if ownership == 'Non-Profit'   else 0,
        'ruca_Metropolitan' : 1 if location  == 'Metropolitan' else 0,
        'ruca_Micropolitan' : 1 if location  == 'Micropolitan' else 0,
        'ruca_Small Town'   : 1 if location  == 'Small Town'   else 0,
        'Data_Year'         : year,
    }])[RQ2_FEATURES]


def build_rq3_row(hospital_ccn, drg_code, bed_count, ownership, location, year,
                   drg_wt_val, log_tot_dschrgs):
    return pd.DataFrame([{
        'DRG_Weight'          : drg_wt_val,
        'BED_CNT'             : bed_count,
        'Log_Tot_Dschrgs'     : log_tot_dschrgs,
        'own_For-Profit'      : 1 if ownership == 'For-Profit'   else 0,
        'own_Non-Profit'      : 1 if ownership == 'Non-Profit'   else 0,
        'ruca_Metropolitan'   : 1 if location  == 'Metropolitan' else 0,
        'ruca_Micropolitan'   : 1 if location  == 'Micropolitan' else 0,
        'ruca_Small Town'     : 1 if location  == 'Small Town'   else 0,
        'Data_Year'           : year,
        'outlier_payment_flag': 0,
    }])[RQ3_FEATURES]


def get_signals(hospital_ccn, drg_code):
    hosp_te_val = float(hosp_te_lookup.get(hospital_ccn, avg_hosp_te))
    drg_te_val  = float(drg_te_lookup.get(drg_code, avg_drg_te))
    drg_wt_vals = df[df['DRG_Cd'] == drg_code]['DRG_Weight']
    drg_wt_val  = float(drg_wt_vals.mean()) if len(drg_wt_vals) > 0 else avg_drg_wt
    hosp_known  = hospital_ccn in hosp_te_lookup.index
    drg_known   = drg_code in drg_te_lookup.index
    hist_dschrg = df[(df['Rndrng_Prvdr_CCN'] == hospital_ccn) &
                     (df['DRG_Cd'] == drg_code)]['Tot_Dschrgs'].mean()
    if np.isnan(hist_dschrg):
        hist_dschrg = df['Tot_Dschrgs'].mean()
    return hosp_te_val, drg_te_val, drg_wt_val, hosp_known, drg_known, hist_dschrg


def run_predictions(hospital_ccn, drg_code, bed_count, ownership, location, year):
    hosp_te_val, drg_te_val, drg_wt_val, hosp_known, drg_known, hist_dschrg = \
        get_signals(hospital_ccn, drg_code)

    rq2_row = build_rq2_row(hospital_ccn, drg_code, bed_count, ownership, location,
                              year, hosp_te_val, drg_te_val, drg_wt_val)
    rq3_row = build_rq3_row(hospital_ccn, drg_code, bed_count, ownership, location,
                              year, drg_wt_val, np.log1p(hist_dschrg))

    pred_dschrg = int(np.expm1(rq2_model.predict(rq2_row)[0]))
    pred_usd    = float(np.expm1(rq3_model.predict(rq3_row)[0]))

    return pred_dschrg, pred_usd, hosp_te_val, drg_te_val, drg_wt_val, hosp_known, drg_known


print('✅ Helper functions ready')

✅ Helper functions ready


In [5]:
# ============================================================
# CELL 5 — Decision Support Tool
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Clear any previous widget output from cell re-runs
clear_output(wait=False)

# Close older widget instances left behind by notebook re-runs
for _widget_name in ["w_ccn", "w_drg", "w_year", "w_billed", "w_beds", "w_own", "w_loc", "w_btn", "out_all", "ui"]:
    _widget = globals().get(_widget_name)
    if hasattr(_widget, "close"):
        _widget.close()

# ── CSS ─────────────────────────────────────────────────────
display(HTML("""<style>
.tool-title{background:linear-gradient(135deg,#0a1628,#1a3a5c);color:white;
  padding:18px 24px;border-radius:10px;border-left:5px solid #2196F3;margin-bottom:12px;}
.tool-title h2{margin:0;font-size:1.4rem;}
.tool-title p{margin:4px 0 0;color:#90b4d4;font-size:0.83rem;}
.sh{font-size:1rem;font-weight:700;color:#1565C0;border-bottom:2px solid #1565C0;
  padding-bottom:3px;margin:12px 0 7px;}
.kpi-row{display:flex;gap:10px;flex-wrap:wrap;margin:8px 0;}
.kpi{background:#f0f4f9;border-radius:8px;padding:9px 14px;flex:1;
  min-width:120px;border-top:3px solid #2196F3;}
.kpi.g{border-top-color:#2E7D32;} .kpi.o{border-top-color:#EF6C00;}
.kpi-l{font-size:0.75rem;color:#607080;text-transform:uppercase;font-weight:700;margin-bottom:2px;}
.kpi-v{font-size:1.6rem;font-weight:700;color:#0d1f35;font-family:monospace;}
.kpi-s{font-size:0.75rem;color:#607080;margin-top:1px;}
.ab{background:#e8f5e9;border:1.5px solid #2E7D32;border-radius:8px;
  padding:8px 12px;margin:5px 0;font-size:0.87rem;}
.mb{background:#fff8e1;border:1.5px solid #f9a825;border-radius:8px;
  padding:8px 12px;margin:5px 0;font-size:0.87rem;color:#4a3700;}
.db{background:#e3f2fd;border-left:4px solid #1565C0;border-radius:0 8px 8px 0;
  padding:6px 12px;font-size:0.88rem;margin:3px 0;}
.fok{background:#e8f5e9;border:2px solid #2E7D32;border-radius:10px;padding:12px 16px;margin:8px 0;}
.fbd{background:#fce4ec;border:2px solid #C62828;border-radius:10px;padding:12px 16px;margin:8px 0;}
.fn{background:#f5f5f5;border:1.5px dashed #90a4ae;border-radius:10px;
  padding:12px 16px;margin:8px 0;color:#607080;text-align:center;}
.ft{font-size:1.05rem;font-weight:800;margin-bottom:5px;}
.ins{background:#e8f4fd;border-left:4px solid #1565C0;border-radius:0 8px 8px 0;
  padding:7px 12px;font-size:0.88rem;margin:5px 0;}
</style>"""))

display(HTML("""<div class='tool-title'>
  <h2>Medicare Decision-Support Tool</h2>
  <p>Discharge Forecast (RQ2) + Payment Benchmark (RQ3) + Billing Outlier Check</p>
</div>"""))

# ── Widgets — created once ───────────────────────────────────
w_ccn = widgets.Combobox(
    placeholder="Type CCN e.g. 370781",
    options=[str(h) for h in hospital_options],
    description="Hospital CCN:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="320px"),
    ensure_option=False, value="370781"
)
w_drg = widgets.Combobox(
    placeholder="Type DRG e.g. 470",
    options=[str(d) for d in drg_options],
    description="DRG Code:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="320px"),
    ensure_option=False, value="470"
)
w_year = widgets.Dropdown(
    options=[2024,2023,2022], value=2024,
    description="Year:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="240px")
)
w_billed = widgets.BoundedFloatText(
    value=0.0, min=0.0, max=9_999_999.0, step=500.0,
    description="Billed Amount $:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="320px")
)
w_beds = widgets.IntSlider(
    value=250, min=10, max=3000, step=10,
    description="Beds:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="380px")
)
w_own = widgets.RadioButtons(
    options=["Non-Profit","For-Profit","Government"],
    value="Non-Profit", description="Ownership:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="300px")
)
w_loc = widgets.RadioButtons(
    options=["Metropolitan","Micropolitan","Small Town","Rural"],
    value="Metropolitan", description="Geography:",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="300px")
)
w_btn = widgets.Button(
    description="Run Analysis",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px")
)
out_all = widgets.Output()

# ── Static layout — render once ──────────────────────────────
ui = widgets.VBox([
    widgets.HTML("<div class='sh'>Step 1 — Enter Hospital &amp; Diagnosis</div>"),
    widgets.HBox([
        widgets.VBox([w_ccn, w_drg, w_year, w_billed,
                      widgets.HTML("<p style='font-size:0.76rem;color:#607080;margin:0;'>"
                                   "Billed amount optional — used for Panel 3. Leave at 0 to skip.</p>")]),
        widgets.HTML("&nbsp;&nbsp;&nbsp;"),
        widgets.VBox([w_beds, w_own, w_loc],
                     layout=widgets.Layout(border="1px solid #ddd", padding="10px",
                                           border_radius="8px", margin="0"))
    ]),
    widgets.HTML("<hr style='margin:10px 0'>"),
    w_btn,
    out_all,
    widgets.HTML("<p style='color:#607080;font-size:0.8rem;margin-top:5px;'>"
                 "Fill in CCN and DRG above then click Run Analysis</p>"),
])
display(ui)


# ── Single click handler — NO observers ─────────────────────
def on_run(b, _out=out_all):
    with _out:
        clear_output(wait=False)

        # Parse
        try:
            ccn      = int(str(w_ccn.value).strip())
            drg_code = int(str(w_drg.value).strip())
        except ValueError:
            display(HTML("<p style='color:red'>Please enter valid CCN and DRG numbers.</p>"))
            return

        year        = int(w_year.value)
        billed_amt  = float(w_billed.value)
        drg_name    = drg_desc_lookup.get(drg_code, "Unknown DRG")

        # Hospital attributes — auto or manual
        if ccn in hosp_lookup:
            h         = hosp_lookup[ccn]
            ownership = h["Ownership_Type"]
            location  = h["RUCA_Group"]
            bed_count = int(h["BED_CNT"])
            src_lbl   = "auto-filled from dataset"
            display(HTML(
                f"<div class='ab'><b>Hospital {ccn} found — auto-filled:</b> "
                f"<b>{ownership}</b> | <b>{location}</b> | <b>{bed_count} beds</b></div>"
            ))
        else:
            ownership = str(w_own.value)
            location  = str(w_loc.value)
            bed_count = int(w_beds.value)
            src_lbl   = "manually entered"
            display(HTML(
                f"<div class='mb'>Hospital {ccn} not in dataset — using manually entered details.</div>"
            ))

        # DRG name
        display(HTML(f"<div class='db'><b>DRG {drg_code}:</b> {drg_name}</div>"))

        # Signals
        ht  = float(hosp_te_lookup.get(ccn, avg_hosp_te))
        dt  = float(drg_te_lookup.get(drg_code, avg_drg_te))
        dw_s= df[df["DRG_Cd"]==drg_code]["DRG_Weight"]
        dw  = float(dw_s.mean()) if len(dw_s)>0 else avg_drg_wt
        hk  = ccn in hosp_te_lookup.index
        dk  = drg_code in drg_te_lookup.index
        hd  = df[(df["Rndrng_Prvdr_CCN"]==ccn)&(df["DRG_Cd"]==drg_code)]["Tot_Dschrgs"].mean()
        if np.isnan(hd): hd = df["Tot_Dschrgs"].mean()

        # Feature rows
        r2 = pd.DataFrame([{
            "DRG_Weight":dw,"BED_CNT":bed_count,"hosp_te":ht,"drg_te":dt,
            "own_For-Profit":1 if ownership=="For-Profit" else 0,
            "own_Non-Profit":1 if ownership=="Non-Profit" else 0,
            "ruca_Metropolitan":1 if location=="Metropolitan" else 0,
            "ruca_Micropolitan":1 if location=="Micropolitan" else 0,
            "ruca_Small Town":1 if location=="Small Town" else 0,
            "Data_Year":year,
        }])[RQ2_FEATURES]
        r3 = pd.DataFrame([{
            "DRG_Weight":dw,"BED_CNT":bed_count,
            "Log_Tot_Dschrgs":np.log1p(hd),
            "own_For-Profit":1 if ownership=="For-Profit" else 0,
            "own_Non-Profit":1 if ownership=="Non-Profit" else 0,
            "ruca_Metropolitan":1 if location=="Metropolitan" else 0,
            "ruca_Micropolitan":1 if location=="Micropolitan" else 0,
            "ruca_Small Town":1 if location=="Small Town" else 0,
            "Data_Year":year,"outlier_payment_flag":0,
        }])[RQ3_FEATURES]

        pd_  = int(np.expm1(rq2_model.predict(r2)[0]))
        pu_  = float(np.expm1(rq3_model.predict(r3)[0]))
        cl   = max(0, pd_ - int(pd_*MAPE_RQ2))
        ch   = pd_ + int(pd_*MAPE_RQ2)
        pl   = pu_*(1-MAPE_RQ3)
        ph   = pu_*(1+MAPE_RQ3)

        # ── PANEL 1 ──────────────────────────────────────
        display(HTML("<div class='sh'>Panel 1 — Discharge Volume Forecast (RQ2)</div>"))
        display(HTML(
            f"<div class='kpi-row'>"
            f"<div class='kpi'><div class='kpi-l'>Predicted Discharges</div>"
            f"<div class='kpi-v'>{pd_:,}</div>"
            f"<div class='kpi-s'>Hospital {ccn} &middot; DRG {drg_code} &middot; {year}</div></div>"
            f"<div class='kpi o'><div class='kpi-l'>Expected Range</div>"
            f"<div class='kpi-v'>{cl}&ndash;{ch}</div>"
            f"<div class='kpi-s'>&plusmn;MAPE ({MAPE_RQ2*100:.0f}%) interval</div></div>"
            f"<div class='kpi'><div class='kpi-l'>Avg Error</div>"
            f"<div class='kpi-v'>&plusmn;{MAE_RQ2}</div>"
            f"<div class='kpi-s'>discharges (MAE)</div></div>"
            f"<div class='kpi g'><div class='kpi-l'>Model R&sup2;</div>"
            f"<div class='kpi-v'>0.73</div>"
            f"<div class='kpi-s'>XGBoost &middot; Temporal split</div></div>"
            f"</div>"
            f"<p style='font-size:0.78rem;color:#607080;margin:2px 0;'>"
            f"{'Known hospital' if hk else 'New hospital — avg fallback'} | "
            f"{'Known DRG' if dk else 'New DRG — avg fallback'} | "
            f"DRG weight: {dw:.3f}</p>"
        ))
        hv = df[df["DRG_Cd"]==drg_code]["Tot_Dschrgs"]
        if len(hv)>0:
            fig,ax = plt.subplots(figsize=(9,3))
            ax.hist(hv.clip(upper=hv.quantile(0.99)),bins=40,
                    color="#90caf9",alpha=0.75,edgecolor="white",label="Historical (2017-2023)")
            ax.axvline(pd_,color="#1565C0",linewidth=2.5,label=f"Forecast: {pd_:,}")
            ax.axvspan(cl,ch,alpha=0.15,color="#1565C0",label=f"Range: {cl}-{ch}")
            ax.set_title(f"DRG {drg_code} — Forecast vs Historical Distribution",
                         fontsize=11,fontweight="bold")
            ax.set_xlabel("Discharges per Hospital-DRG pair")
            ax.set_ylabel("Count of records")
            ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
            ax.grid(axis="x",alpha=0.2); fig.tight_layout(); display(fig); plt.close(fig)

        # ── PANEL 2 ──────────────────────────────────────
        display(HTML("<div class='sh'>Panel 2 — Expected Medicare Payment (RQ3)</div>"))
        display(HTML(
            f"<p style='font-size:0.83rem;color:#455a64;margin:0 0 5px;'>"
            f"DRG <b>{drg_code} — {drg_name}</b> | {ownership} &middot; {location} &middot; {bed_count} beds</p>"
            f"<div class='kpi-row'>"
            f"<div class='kpi g'><div class='kpi-l'>Expected Payment</div>"
            f"<div class='kpi-v'>${pu_:,.0f}</div>"
            f"<div class='kpi-s'>per discharge</div></div>"
            f"<div class='kpi'><div class='kpi-l'>Typical Range</div>"
            f"<div class='kpi-v'>${pl:,.0f}&ndash;${ph:,.0f}</div>"
            f"<div class='kpi-s'>&plusmn;MAPE ({MAPE_RQ3*100:.0f}%) interval</div></div>"
            f"<div class='kpi o'><div class='kpi-l'>Avg Error</div>"
            f"<div class='kpi-v'>&plusmn;${MAE_RQ3:,.0f}</div>"
            f"<div class='kpi-s'>per discharge (MAE)</div></div>"
            f"<div class='kpi g'><div class='kpi-l'>Model R&sup2;</div>"
            f"<div class='kpi-v'>0.92</div>"
            f"<div class='kpi-s'>XGBoost &middot; Random split</div></div>"
            f"</div>"
        ))

        # ── PANEL 3 ──────────────────────────────────────
        display(HTML("<div class='sh'>Panel 3 — Billing Outlier Check</div>"))
        if billed_amt > 0:
            dd  = df[df["DRG_Cd"]==drg_code]
            n   = len(dd)
            bp25= float(dd["Avg_Submtd_Cvrd_Chrg"].quantile(0.25)) if n>0 else 0
            bmd = float(dd["Avg_Submtd_Cvrd_Chrg"].median())       if n>0 else 0
            bp75= float(dd["Avg_Submtd_Cvrd_Chrg"].quantile(0.75)) if n>0 else 0
            pp25= float(dd["Avg_Mdcr_Pymt_Amt"].quantile(0.25))    if n>0 else 0
            pmd = float(dd["Avg_Mdcr_Pymt_Amt"].median())          if n>0 else 0
            pp75= float(dd["Avg_Mdcr_Pymt_Amt"].quantile(0.75))    if n>0 else 0
            gmd = float(dd["Payment_Gap"].median())                 if n>0 else 0
            rp25= float(dd["Payment_Ratio"].quantile(0.25))        if n>0 else 0.13
            rmd = float(dd["Payment_Ratio"].median())              if n>0 else 0.20
            rp75= float(dd["Payment_Ratio"].quantile(0.75))        if n>0 else 0.30
            tr  = pu_/billed_amt
            tg  = billed_amt - pu_
            bl  = billed_amt < bp25; bh = billed_amt > bp75
            rl  = tr < rp25;         rh = tr > rp75
            af  = bl or bh or rl or rh

            def st(v,lo,hi):
                if v<lo:   return "BELOW NORMAL","#C62828"
                elif v>hi: return "ABOVE NORMAL","#EF6C00"
                return "NORMAL","#2E7D32"

            bst,bc_ = st(billed_amt,bp25,bp75)
            rst,rc_ = st(tr,rp25,rp75)
            fc  = "fbd" if af else "fok"
            ft_ = "FLAGS DETECTED — Review recommended" if af else "ALL METRICS NORMAL"
            fc_ = "#C62828" if af else "#2E7D32"

            display(HTML(
                f"<div class='{fc}'>"
                f"<div class='ft' style='color:{fc_};'>{ft_}</div>"
                f"<p style='font-size:0.83rem;margin:2px 0;'>DRG {drg_code} — {drg_name}</p>"
                f"</div>"
            ))
            display(HTML(
                f"<table style='width:100%;border-collapse:collapse;font-size:0.87rem;margin:8px 0;'>"
                f"<thead><tr style='background:#0d1f35;color:white;'>"
                f"<th style='padding:7px 10px;text-align:left;'>Metric</th>"
                f"<th style='padding:7px 10px;text-align:right;'>This Hospital</th>"
                f"<th style='padding:7px 10px;text-align:right;'>National 25th</th>"
                f"<th style='padding:7px 10px;text-align:right;'>National Median</th>"
                f"<th style='padding:7px 10px;text-align:right;'>National 75th</th>"
                f"<th style='padding:7px 10px;text-align:center;'>Status</th>"
                f"</tr></thead><tbody>"
                f"<tr style='background:#f9f9f9;border-bottom:1px solid #eee;'>"
                f"<td style='padding:7px 10px;font-weight:600;'>Hospital billed</td>"
                f"<td style='padding:7px 10px;text-align:right;font-weight:700;'>${billed_amt:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${bp25:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${bmd:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${bp75:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:center;font-weight:700;color:{bc_};'>{bst}</td>"
                f"</tr>"
                f"<tr style='border-bottom:1px solid #eee;'>"
                f"<td style='padding:7px 10px;font-weight:600;'>Expected Medicare payment</td>"
                f"<td style='padding:7px 10px;text-align:right;font-weight:700;'>${pu_:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${pp25:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${pmd:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${pp75:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:center;color:#1565C0;font-weight:600;'>PREDICTED</td>"
                f"</tr>"
                f"<tr style='background:#f9f9f9;border-bottom:1px solid #eee;'>"
                f"<td style='padding:7px 10px;font-weight:600;'>Payment gap (billed minus paid)</td>"
                f"<td style='padding:7px 10px;text-align:right;font-weight:700;'>${tg:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>—</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>${gmd:,.0f}</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>—</td>"
                f"<td style='padding:7px 10px;text-align:center;color:#607080;'>—</td>"
                f"</tr>"
                f"<tr style='border-bottom:1px solid #eee;'>"
                f"<td style='padding:7px 10px;font-weight:600;'>Payment ratio (Medicare per $1 billed)</td>"
                f"<td style='padding:7px 10px;text-align:right;font-weight:700;color:{rc_};'>{tr*100:.1f}%</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>{rp25*100:.1f}%</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>{rmd*100:.1f}%</td>"
                f"<td style='padding:7px 10px;text-align:right;color:#607080;'>{rp75*100:.1f}%</td>"
                f"<td style='padding:7px 10px;text-align:center;font-weight:700;color:{rc_};'>{rst}</td>"
                f"</tr>"
                f"</tbody></table>"
            ))
            msgs = []
            if bl: msgs.append((f"Billed <b>${billed_amt:,.0f}</b> is below the normal range for DRG {drg_code} (${bp25:,.0f}–${bp75:,.0f}). Hospital may have <b>underbilled</b>.","#C62828"))
            if bh: msgs.append((f"Billed <b>${billed_amt:,.0f}</b> is above the normal range for DRG {drg_code} (${bp25:,.0f}–${bp75:,.0f}). High billing — may attract audit scrutiny.","#EF6C00"))
            if rl: msgs.append((f"Payment ratio <b>{tr*100:.1f}%</b> is below the 25th percentile ({rp25*100:.1f}%) for DRG {drg_code}. Medicare paying unusually small share.","#C62828"))
            if rh: msgs.append((f"Payment ratio <b>{tr*100:.1f}%</b> is above the 75th percentile ({rp75*100:.1f}%) for DRG {drg_code}. Likely caused by underbilling.","#EF6C00"))
            if not af: msgs.append((f"All metrics within normal range for DRG {drg_code}. Billed amount and payment ratio are both typical.","#2E7D32"))
            for msg,col in msgs:
                display(HTML(f"<div class='ins' style='border-left-color:{col};'>{msg}</div>"))
        else:
            display(HTML(
                "<div class='fn'><b>Enter billed amount above to activate Panel 3</b><br>"
                "Compares your billed amount and predicted payment ratio against national averages for this DRG.</div>"
            ))

        # ── Summary ──────────────────────────────────────
        display(HTML("<div class='sh'>Full Summary</div>"))
        rows = [
            ["Hospital CCN", ccn],
            ["Hospital", f"{ownership} | {location} | {bed_count} beds ({src_lbl})"],
            ["DRG", f"{drg_code} — {drg_name}"],
            ["Year", year],
            ["DRG weight", f"{dw:.3f}"],
            ["Predicted discharges", f"{pd_:,} (range: {cl}–{ch})"],
            ["Expected Medicare payment", f"${pu_:,.0f} (range: ${pl:,.0f}–${ph:,.0f})"],
        ]
        if billed_amt > 0:
            rows += [
                ["Hospital billed", f"${billed_amt:,.0f}"],
                ["Payment gap", f"${tg:,.0f}"],
                ["Payment ratio", f"{tr*100:.1f}%"],
                ["Outlier flag", "YES — review recommended" if af else "NO — all normal"],
            ]
        display(pd.DataFrame(rows,columns=["Item","Value"])
                  .style.set_properties(**{"text-align":"left","font-size":"13px"})
                  .hide(axis="index"))

w_btn.on_click(on_run)


---
### How to use this tool

1. **Hospital CCN** — type any CCN from the dataset (e.g. `370781`)
2. **DRG Code** — type any DRG (e.g. `470` = Major Hip/Knee Replacement)
3. **Beds, Year, Ownership, Geography** — select from the controls
4. **Actual Payment** — optional. Enter what CMS actually paid to trigger the outlier check
5. Click **Run Analysis**

### What each panel shows

| Panel | Shows |
|---|---|
| 1 — Discharge Forecast | How many patients expected + where that sits in historical distribution |
| 2 — Payment Prediction | What Medicare should pay per discharge |
| 3 — Outlier Check | 🔴 if actual payment is >$2,190 below expected — flags for investigation |